# **Diplomado IA: Aplicaciones de Inteligencia Artificial I - Parte 2**. <br> Práctico 22: Generación de resúmenes
---
---

**Profesores:**
- Felipe del Río

**Ayudante:**
- Bianca del Solar
---
---

# **Instrucciones Generales**

El siguiente práctico se debe realizar de forma **individual**. El formato de entregar es el **archivo .ipynb con todas las celdas ejecutadas**. Todas las preguntas deben ser respondida en celdas de texto. No se aceptará el _output_ de una celda de código como respuesta.

**Nombre:** Roberto Araneda

El siguiente práctico cuanta con 3 secciones donde cada una contendrá 1 o más actividades a realizar. Algunas actividades correspondrán a escribir código y otras a responder preguntas.

**Importante.** Para facilitar su ejecución, cada sección puede ser ejecutada independientemente.

Se recomienda **fuertemente** revisar las secciones donde se entrega código porque algunas actividades de código pueden reutilizar el mismo código pero con cambios en algunas líneas.



# **Agenda**

>[Diplomado IA: Aplicaciones de Inteligencia Artificial I - Parte 2.  Práctico 4: Generación de resúmenes](#scrollTo=tHopPtVaNF1K)

>[Instrucciones Generales](#scrollTo=uIdAKAdELPSl)

>[Agenda](#scrollTo=kEloa5uXLIPK)

>[Parte I: Generación de Resumenes Extractivos](#scrollTo=Ww8OOQyAnBQj)

>>[Preámbulo](#scrollTo=jdQWS0upPEir)

>>>[Funciones Auxiliares](#scrollTo=UBo9B9xujjlr)

>>[Modelo de resumenes extractivo](#scrollTo=7b6hqPsgPKhM)

>>>[Entrenamiento](#scrollTo=Ge3EMmSDSut6)

>>>[Configuración de nuestra ejecución](#scrollTo=3Cw_GQPmgsul)

>>>[Datos](#scrollTo=dV73kM9nihPx)

>>>[Evaluación](#scrollTo=l5hju-cES9hD)

>>>[Actividad 1](#scrollTo=RGuwGZGM0Y12)



# Parte I: Generación de Resumenes Extractivos

En esta actividad aplicaremos lo que vimos acerca de métodos para resumir de forma extractiva. Nos basaremos en un modelo que sigue el proceso planteado para modelos extractivos que vimos en clase. Este modelo viene del trabajo [Fine-tune BERT for Extractive Summarization](https://arxiv.org/pdf/1903.10318.pdf), a continuación se detalla un diagrama mostrando la forma en que funciona este modelo.

<figure>
<center>
<img src='https://docs.google.com/drawings/d/e/2PACX-1vSa16s1rK-JLF_E13C5E4vLe6qRp7azTi-7tLXpUTVisr-OUntDVs-nezE4sWH4QCOLPdwf_JCxvIut/pub?w=765&h=494' height="250" />
<figcaption>Figura 1: Modelo Extractivo.</figcaption>
</center>

</figure>

Por el momento no entraremos en detalle de la arquitectura de *transformer*, pues lo verán en el futuro. Pero si les gana la curiosidad, les recomiendo leer el paper que los propuso, ["Attention is all you need"](https://papers.nips.cc/paper/7181-attention-is-all-you-need.pdf) en NeurIPS 2017, y lo mismo con el paper de [BERT](https://arxiv.org/pdf/1810.04805.pdf) publicado en ACL 2019.

Este modelo de lenguaje será visto en la próxima sección del diplomado.


<small>Basado en la [implementación oficial](https://github.com/nlpyang/BertSum) de este trabajo.</small>


## Preámbulo
Primero debemos copiar el dataset y otros archivos que utilizaremos hacia colab. El dataset lo decomprimiremos dentro del directorio data, que recién creamos. También instalaremos algunas de las librerías que ocuparemos durante esta actividad.

In [1]:
import warnings
warnings.filterwarnings('ignore')

Primero que todo descargaremos las librerías que vamos a utilizar, en este caso debemos modificar la versión de PyTorch que utilizaremos a la `1.1.0`. Esto se debe a que la implementación original utilizó esa versión y así mantenmos mejor compatibilidad.

In [2]:
!pip install -qqq --force-reinstall urllib3 folium # Nos evitan mensajes de errores, aunq que no influyen en la ejecución

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.4/113.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.1/94.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.1/134.1 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requir

In [3]:
!pip install -qqq pytorch_pretrained_bert tensorboardX pyrouge

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.7/86.7 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.5/60.5 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.8/123.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 8.6 MB/s eta 0:00:00


In [4]:
![ ! -d "/content/BertSum" ] && git clone --progress https://github.com/fdelrio89/BertSum.git

Cloning into 'BertSum'...
remote: Enumerating objects: 306, done.
remote: Counting objects: 100% (298/298), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 306 (delta 170), reused 289 (delta 164), pack-reused 8 (from 1)
Receiving objects: 100% (306/306), 15.05 MiB | 18.11 MiB/s, done.
Resolving deltas: 100% (170/170), done.


Debemos también descargar los archivos del modelo pre-entrenado.

In [5]:
!pip3 install -qqq --upgrade gdown

In [6]:
import os
import random

if not os.path.exists('/content/lab4.zip'):
    if random.random() > 0.5:
        !gdown --no-cookies 18ecT1r9jFGyChATPwTdRgvj9dKctd3sS -O /content/lab4.zip
    else:
        !gdown --no-cookies 1tGez67YBFkZqY3qyCCul1Wex96PKkeqo -O /content/lab4.zip

Downloading...
From (original): https://drive.google.com/uc?id=18ecT1r9jFGyChATPwTdRgvj9dKctd3sS
From (redirected): https://drive.google.com/uc?id=18ecT1r9jFGyChATPwTdRgvj9dKctd3sS&confirm=t&uuid=d3f4a31d-660a-48cf-8211-e45340e9eab2
To: /content/lab4.zip
100% 2.08G/2.08G [00:30<00:00, 67.6MB/s]


También debemos descargar los datos en donde probaremos nuestros modelos.

In [7]:
![ ! -d "/content/lab4" ] && unzip -nq lab4.zip -d .
!mv lab4/activity\ 1/model.pt BertSum/models/model.pt

In [8]:
if not os.path.exists('/content/bertsum_data.zip'):
    !gdown 1x0d61LP9UAN389YN00z0Pv-7jQgirVg6 -O /content/bertsum_data.zip

!unzip -nq bertsum_data.zip -d BertSum/bert_data

Downloading...
From (original): https://drive.google.com/uc?id=1x0d61LP9UAN389YN00z0Pv-7jQgirVg6
From (redirected): https://drive.google.com/uc?id=1x0d61LP9UAN389YN00z0Pv-7jQgirVg6&confirm=t&uuid=035a6e97-395f-4d05-9e95-4b7d71f7fb97
To: /content/bertsum_data.zip
100% 869M/869M [00:14<00:00, 59.8MB/s]


In [9]:
os.chdir('/content/BertSum/src/')

Finalmente configuraremos nuestra ejecución.

In [10]:
class Args():
    encoder='classifier'                        # Tipo de encoder a utilizar
    mode='test'                                 # Testearemos el modelo, no lo entrenaremos
    bert_data_path='../bert_data/cnndm'         # Path a los datos
    model_path='../models/'                     # Path a los checkpoints
    result_path='../results/cnndm'
    temp_dir='../temp'
    bert_config_path='../bert_config_uncased_base.json' # Archivo de configuración de hugginface de BERT
    batch_size=1000
    use_interval=True
    hidden_size=128
    ff_size=512
    heads=4
    inter_layers=2
    rnn_size=512
    param_init=0
    param_init_glorot=True
    dropout=0.1
    optim='adam'                                # Optimizador a utilizar
    lr=1                                        # Tasa de aprendizaje
    beta1= 0.9
    beta2=0.999
    decay_method=''
    warmup_steps=8000
    max_grad_norm=0
    save_checkpoint_steps=5
    accum_count=1
    world_size=1
    report_every=1
    train_steps=1000
    recall_eval=False
    visible_gpus=-1
    gpu_ranks=[0]
    log_file='../logs/cnndm.log'
    dataset=''
    seed=1666
    test_all=False
    test_from=''
    train_from=''
    report_rouge=False
    block_trigram=True

### Funciones Auxiliares

Definamos algunas funciones que nos ayudarán a visualizar sus resultados de mejor forma.

In [11]:
import textwrap

# Auxiliary function to help in the displaying of results
def bold(text):
  return '\033[1m' + text + '\033[0m'

def highlight(text):
  return '\033[43m' + text + '\033[0m'

def display_translation(srcs, tgts, preds=None):
    bold_end = bold('<sent_end>')
    preds = [None] * len(srcs) if preds is None else preds
    for idx, (src, tgt, pred) in enumerate(zip(srcs, tgts, preds), start=1):
        print(bold(f'Sample {idx}'))
        print(bold('Source Text:'))
        for sent in src:
            sent = highlight(sent) if (pred is not None) and (sent in pred.split('\n')) else sent
            print(textwrap.fill(sent, width=70), end=f' {bold_end}\n')
        print()

        print(bold('Gold Standard Summarization:'))
        # print(textwrap.fill(tgt, width=70), end=' {bold_end}\n\n')
        for sent in tgt.split('<q>'):
            print(textwrap.fill(sent, width=70), end=f' {bold_end}\n')
        print()

        if pred is not None:
            print(bold('Predicted Summarization:'))
            # for sent in pred:
            # print(textwrap.fill(pred, width=70), end=f' {bold_end}\n\n')
            for sent in pred.split('\n'):
                print(textwrap.fill(sent, width=70), end=f' {bold_end}\n')
            print()

        print('                 ===========================                  \n')

# Evaluation
def _get_ngrams(n, text):
    ngram_set = set()
    text_length = len(text)
    max_index_ngram_start = text_length - n
    for i in range(max_index_ngram_start + 1):
        ngram_set.add(tuple(text[i:i + n]))
    return ngram_set

def _block_tri(c, p):
    tri_c = _get_ngrams(3, c.split())
    for s in p:
        tri_s = _get_ngrams(3, s.split())
        if len(tri_c.intersection(tri_s))>0:
            return True
    return False

## Modelo de resumenes extractivo

En esta sección tomaremos un modelo para generar resumenes extractivos pre-entrenado y veremos como funciona en el dataset de resumenes de noticias CNN/DailyMail.

En esta actividad no entrenaremos un modelo, de todas maneras está disponible el código para que uds entrenen su propio modelo en caso de que así lo deseen.

Partimos importando los paquetes que utilizaremos y utilizando las configuraciones que seteamos previamente.

In [12]:
import numpy as np
import torch
from pytorch_pretrained_bert import BertConfig

from models import data_loader
from models.data_loader import load_dataset
from models.model_builder import Summarizer
from models.trainer import build_trainer

self_trained = False

### Entrenamiento

Para poder entrenar su propio modelo deben descomentar la siguientes **dos celdas** de código.

Si se fijan bien, estamos configurando nuestro entrenamiento con los argumentos que le pasamos al script al llamar a `python train.py`. En base a estos argumentos pueden modificar el modelo y el entrenamiento mismo, sin embargo es importante que seteen correctamente los siguientes parámetros. Aun que si corren como está por defecto no deberían tener problemas.

- `bert_data_path`: Path a los datos de entrenamiento, estos deben seguir el formato indicado en [esta sección](https://github.com/nlpyang/BertSum/#data-preparation-for-cnndailymail) del repositorio.
- `model_path`: Path a donde serán guardados los checkpoints del entrenamiento.
- `log_file`: Path al log del entrenamiento.
- `batch_size`: El tamaño del batch, si tienen problemas de memoria de la GPU (no deberían en colab) pueden disminuírlo.
- `train_steps`: Número de iteraciones del entrenamiento, el entrnemineto original consta de 50.000 iteraciones, pero pueden probar con menos. Para este caso también pueden setearlo en la variable de entorno `$TRAIN_STEPS`


De todas maneras pueden encontrar más información en el [repositorio del proyecto](https://github.com/nlpyang/BertSum/#model-training).

Es importante notar que el entrenamiento durará un par de horas en caso de que quieran realizarlo.

In [ ]:
# self_trained = True

In [ ]:
# %%shell
# TRAIN_STEPS=1000 # Entrenamiento corto de prueba, originalmente 50.000
# python train.py -mode train -encoder classifier -dropout 0.1 -bert_data_path ../bert_data/cnndm \
#                 -model_path ../models/bert_classifier -lr 2e-3 -visible_gpus 0  -gpu_ranks 0 \
#                 -world_size 1 -report_every 50 -save_checkpoint_steps 1000 -batch_size 1000 \
#                 -decay_method noam -train_steps $TRAIN_STEPS -accum_count 2 -log_file ../logs/bert_classifier \
#                 -use_interval true -warmup_steps 10000
# mv ../models/bert_classifier/model_step_${TRAIN_STEPS}.pt ../models/model-self-trained.pt

### Configuración de nuestra ejecución

Para poder configurar correctamente nuestra ejecución utlizaremos los argumentos previamente definidos, para que estos estén ad-hoc a como está dispuesto nuestro entorno de colab. También consideraremos los argumentos que se usaron en el entrenamiento para mantenerlos igual en la creación del modelo.

Estos últimos vienen en el checkpoint que tenemos del entrenamiento, así que lo leemos y actualizamos nuestros argumentos.

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [14]:
model_flags = ['hidden_size', 'ff_size', 'heads', 'inter_layers','encoder','ff_actv', 'use_interval','rnn_size']

args = Args()
checkpoint_path = f'{args.model_path}/model.pt' if not self_trained else f'{args.model_path}/model-self-trained.pt'
checkpoint = torch.load(checkpoint_path, weights_only=False, map_location=lambda storage, loc: storage)

opt = vars(checkpoint['opt'])
for k in opt.keys():
    if (k in model_flags):
        setattr(args, k, opt[k])

### Datos

Para poder evaluar nuestro modelo primero debemos cargar los datos de test. Además podemos aprovechar de ver la forma que tienen estos datos y hacernos una mejor idea de la tarea que está haciendo el modelo.

In [15]:
test_iter = data_loader.Dataloader(args, load_dataset(args, 'test', shuffle=True),
                              args.batch_size, device,
                              shuffle=False, is_test=True)

In [16]:
batch = next(iter(test_iter))
display_translation(batch.src_str, batch.tgt_str)

Sample 1
Source Text:
a lonely shepherd has been found dead alongside a scarecrow he had
apparently had sex with after dressing it up in a long-haired wig and
lipstick . <sent_end>
the rotting remains of jose alberto , 58 , were discovered after
neighbours called their local council to report the smell coming from
his house in the city of san jose de balcare in eastern argentina . <sent_end>
rodolfo moure , a spokesman for the prosecutor 's office , said : ' i
initially thought there were two bodies but then i realised one was a
scarecrow wearing lipstick and a long-haired wig . <sent_end>
jose alberto , 58 , was found dead next to a scarecrow prosecutors
believe he died while having sex with it <sent_end>
` it was lying next to the deceased . <sent_end>
` there were no signs of violence and we are working on the assumption
that the man died during sex with the scarecrow . <sent_end>
` straw had been stuffed inside the old clothes that had been sewn
together to make the scarecrow . <se

### Evaluación

Evaluaremos nuestro modelo cualitativamente. Para esto primero debemos crear nuestro modelo y cargar los pesos que teníamos pre-entrenados.

Utilizaremos la clase `Summarizer` que es a su vez un `nn.Module` de PyTorch. Esta clase integra los distintos módulos que se plantearon anteriormente para crear un resumen en base a los datos entregados.

Finalmente cargamos los pesos utilizando la función `Summarizer.load_cp`, que recibe el mismo `checkpoint` que leímos anteriormente.

In [17]:
config = BertConfig.from_json_file(args.bert_config_path) # configuracion para la librería HuggingFace Transformers
model = Summarizer(args, device, load_pretrained_bert=False, bert_config=config)
model.load_cp(checkpoint)

Ya creado nuestro modelo, podemos ver algunos ejemplos de las predicciones que este hace. Aquí:

- **Source Text:** Corresponde al documento que será resumido. En este caso al tratarse de un modelo extractivo destacamos aquí mismo las frases seleccionadas.
- **Gold Standard Summarization:** Corresponde al resumen de referencia creado para este text, se usa a modo de "ground truth".
- **Predicted Summarization:** Corresponde al resumen generado por nuestro modelo. En este caso al ser un modelo extractivo, es una selección de oraciones del texto original.

In [18]:
model.eval()
with torch.no_grad():
    batch = next(iter(test_iter))

    src = batch.src
    labels = batch.labels
    segs = batch.segs
    clss = batch.clss
    mask = batch.mask
    mask_cls = batch.mask_cls

    gold = []
    pred = []

    sent_scores, mask = model(src, segs, clss, mask, mask_cls)

    sent_scores = sent_scores + mask.float()
    sent_scores = sent_scores.cpu().data.numpy()
    selected_ids = np.argsort(-sent_scores, 1)

    for i, idx in enumerate(selected_ids):
        _pred = []
        if len(batch.src_str[i]) == 0:
            continue
        for j in selected_ids[i][:len(batch.src_str[i])]:
            if j >= len(batch.src_str[i]):
                continue
            candidate = batch.src_str[i][j].strip()
            if args.block_trigram:
                if(not _block_tri(candidate, _pred)):
                    _pred.append(candidate)
            else:
                _pred.append(candidate)

            if ((not args.recall_eval) and len(_pred) == 3):
                break

        _pred = '\n'.join(_pred)
        if (args.recall_eval):
            _pred = ' '.join(_pred.split()[:len(batch.tgt_str[i].split())])

        pred.append(_pred)
        gold.append(batch.tgt_str[i])

display_translation(batch.src_str, gold, pred)

Sample 1
Source Text:
british no 2 aljaz bedene was unable to sustain his strong run since
switching allegiance from slovenia , crashing out of the grand prix
hassan ii in casablanca in the last eight . <sent_end>
the world no 99 had breezed past fellow-qualifier arthur de greef
of belgium in a 70-minute , straight-sets victory to set up his
quarter-final clash - but was unable to dislodge czech young gun jiri
vesely on friday . <sent_end>
vesely saw off bedene 6-1 6-4 to reach the final four and tee up
a clash with spanish veteran daniel gimeno-traver . <sent_end>
aljaz bedene , who has switched from slovenia to great britain ,
lost in casablanca ( not pictured ) <sent_end>
the 21-year-old rising czech star put paid to bedene 's good start to
life under the official british banner , after the london-based
25-year-old changed nationality last month . <sent_end>
vesely saw off mikhail youzhny on thursday despite all his luggage
being lost in transit from rome and borrowing kit from doub

### Actividad 1


Responda **brevemente** las siguientes preguntas sobre el modelo utilizado y la tarea de *extractive summarization*.

In [19]:
#@title Para una oración, ¿cuál es el output del modelo utilizado en la actividad?
R = "Un score escalar en [0,1] (salida sigmoide) por cada oración, que estima la probabilidad de que la oración pertenezca al resumen. No produce texto." #@param {type:"string"}

In [20]:
#@title ¿Cómo se determina en nuestro método qué oraciones irán al resumen final en base a este output?
R = "Se rankean las oraciones por su score descendente, se aplica trigram blocking (no agregar una oración si comparte un trigrama con las ya elegidas) y se toman las 3 primeras que sobrevivan al filtro." #@param {type:"string"}

In [21]:
#@title Proponga una alternativa válida para seleccionar las frases que deben ir en el resumen extractivo, en base al resumen estándar de oro o *gold standard*.
R = "En lugar de solapamiento de n-gramas, usar similitud coseno entre los embeddings (Sentence-BERT) de cada oración del documento y el gold, etiquetando como positivas las de mayor similitud semántica. Captura paráfrasis que ROUGE no detecta." #@param {type:"string"}

Responda verdadero o falso.

In [22]:
#@title Un modelo de este tipo es capaz de generar texto que no aparece en la entrada.
R = "Falso" #@param ["seleccione una opcion", "Verdadero", "Falso"]

In [26]:
#@title Para efectos del modelo la entrada es todo el texto que se quiere resumir.
R = "Verdadero" #@param ["seleccione una opcion", "Verdadero", "Falso"]

In [24]:
#@title Es posible cambiar, fácilmente, el modelo de BERT por una LSTM pre-entrenada en nuestro modelo.
R = "Falso" #@param ["seleccione una opcion", "Verdadero", "Falso"]

In [25]:
#@title Nuestro modelo, antes de ser entrenada para esta tarea, no tuvo un pre-entrenamiento.
R = "Falso" #@param ["seleccione una opcion", "Verdadero", "Falso"]